SparkSession + catálogo Iceberg

In [1]:
import os
import socket
import time
from urllib import error, request
from py4j.protocol import Py4JJavaError
from pyspark.sql import SparkSession


def resolve_service_host(service_name: str, fallback_host: str = "localhost") -> str:
    try:
        socket.gethostbyname(service_name)
        return service_name
    except socket.gaierror:
        return fallback_host


def is_http_ready(url: str, timeout: float = 2.0) -> bool:
    try:
        with request.urlopen(url, timeout=timeout) as _:
            return True
    except error.HTTPError:
        # Si el servicio responde con HTTP error, al menos esta vivo.
        return True
    except Exception:
        return False


iceberg_host = resolve_service_host("iceberg-rest")
minio_host = resolve_service_host("minio")
kafka_host = resolve_service_host("kafka")

default_iceberg_uri = f"http://{iceberg_host}:8181"
default_minio_endpoint = f"http://{minio_host}:{'9000' if minio_host == 'minio' else '9100'}"
default_kafka_bootstrap = f"{kafka_host}:{'29092' if kafka_host == 'kafka' else '9092'}"

ICEBERG_REST_URI = os.getenv("ICEBERG_REST_URI", default_iceberg_uri)
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", default_minio_endpoint)
KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", default_kafka_bootstrap)

print("Endpoints detectados:")
print(f"  ICEBERG_REST_URI={ICEBERG_REST_URI}")
print(f"  MINIO_ENDPOINT={MINIO_ENDPOINT}")
print(f"  KAFKA_BOOTSTRAP_SERVERS={KAFKA_BOOTSTRAP_SERVERS}")

# Espera corta para Iceberg REST al reiniciar contenedores
iceberg_config_url = f"{ICEBERG_REST_URI}/v1/config"
for _ in range(20):
    if is_http_ready(iceberg_config_url):
        break
    time.sleep(2)

# Usa la ruta Ivy tradicional y prepara subcarpetas para resolver dependencias
ivy_dir = "/home/jovyan/.ivy2"
os.makedirs(f"{ivy_dir}/cache", exist_ok=True)
os.makedirs(f"{ivy_dir}/jars", exist_ok=True)

spark = (
    SparkSession.builder
    .appName("bronze-payments-stream")
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0",
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1",
            "org.apache.iceberg:iceberg-aws-bundle:1.6.1"
        ])
    )
    .config("spark.jars.ivy", ivy_dir)
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "rest")
    .config("spark.sql.catalog.lakehouse.uri", ICEBERG_REST_URI)
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint", MINIO_ENDPOINT)
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id", "admin")
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", "admin123")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# Reintento minimo para evitar fallo transitorio de DNS/REST al arrancar desde cero
for attempt in range(1, 11):
    try:
        spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.bronze")
        print("Namespace lakehouse.bronze listo")
        break
    except Py4JJavaError as e:
        message = str(e)
        transient = "UnknownHostException" in message or "RESTException" in message
        if transient and attempt < 10:
            print(f"Reintento {attempt}/10: esperando catalogo Iceberg...")
            time.sleep(3)
            continue
        raise

Endpoints detectados:
  ICEBERG_REST_URI=http://iceberg-rest:8181
  MINIO_ENDPOINT=http://minio:9000
  KAFKA_BOOTSTRAP_SERVERS=kafka:29092
Namespace lakehouse.bronze listo


Lectura Kafka + esquema original

In [2]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema = StructType([
    StructField("event_time", StringType(), True),
    StructField("payment_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("card_id", StringType(), True),
    StructField("merchant_id", StringType(), True),
    StructField("device_id", StringType(), True),
    StructField("ip", StringType(), True),
    StructField("country", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("status", StringType(), True),
    StructField("mcc", StringType(), True),
])

kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", "payment-events")
    .option("startingOffsets", "latest")
    .load()
)

bronze_df = (
    kafka_df
    .selectExpr("CAST(value AS STRING) AS json_payload")
    .select(from_json(col("json_payload"), schema).alias("data"))
    .select("data.*")
)

Escritura streaming a Iceberg Bronze

In [3]:
import time

query = (
    bronze_df.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/chk/bronze_payments_raw")
    .toTable("lakehouse.bronze.payments_raw")
)

print(f"✅ Streaming query iniciado")
print(f"ID Query: {query.id if query else 'N/A'}")
print(f"Nombre: {query.name if query else 'N/A'}")
print(f"Status: {query.status if query else 'No iniciado'}")

# Dejar correr 10 segundos para que ingieran algunos eventos
time.sleep(10)

# Mostrar status
if query:
    print(f"\nStatus después de 10s:")
    print(f"  Activo: {query.isActive}")
    print(f"  Estatísticas: {query.lastProgress}")


✅ Streaming query iniciado
ID Query: de754e50-a7d9-41f7-9093-742162aad247
Nombre: None
Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}

Status después de 10s:
  Activo: True
  Estatísticas: {'id': 'de754e50-a7d9-41f7-9093-742162aad247', 'runId': '934880dd-e874-490b-a64c-14c62cf05e21', 'name': None, 'timestamp': '2026-04-13T11:02:23.236Z', 'batchId': 14, 'numInputRows': 2, 'inputRowsPerSecond': 9.345794392523365, 'processedRowsPerSecond': 11.428571428571429, 'durationMs': {'addBatch': 134, 'commitOffsets': 15, 'getBatch': 0, 'latestOffset': 3, 'queryPlanning': 7, 'triggerExecution': 175, 'walCommit': 16}, 'stateOperators': [], 'sources': [{'description': 'KafkaV2[Subscribe[payment-events]]', 'startOffset': {'payment-events': {'0': 28}}, 'endOffset': {'payment-events': {'0': 30}}, 'latestOffset': {'payment-events': {'0': 30}}, 'numInputRows': 2, 'inputRowsPerSecond': 9.345794392523365, 'processedRowsPerSecond': 11.428571428571429, 'metrics'

Verificación rápida

In [4]:
# DEBUG: Verifica que Kafka tiene eventos
print("=" * 80)
print("DEBUGGING KAFKA")
print("=" * 80)

# Leer eventos raw desde Kafka (sin transformar)
kafka_raw = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", "payment-events")
    .option("startingOffsets", "earliest")
    .load()
)

print("\nTotal de eventos en Kafka:")
count = kafka_raw.count()
print(f"   Cantidad: {count}")

if count > 0:
    print("\nPrimeros 3 eventos raw:")
    kafka_raw.limit(3).select(
        "key",
        "value",
        "partition",
        "offset",
        "timestamp"
    ).show(truncate=False)
else:
    print("   No hay eventos en Kafka. Verifica que scripts/generar_datos.py esta ejecutandose.")

DEBUGGING KAFKA

Total de eventos en Kafka:
   Cantidad: 35

Primeros 3 eventos raw:
+-------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [5]:
# ✅ VERIFICACIÓN FINAL
print("=" * 80)
print("✅ VERIFICACIÓN FINAL DEL PROCESO")
print("=" * 80)

# 1. Estado del streaming query
print("\n📡 Estado del Streaming Query:")
if 'query' in globals():
    print(f"   Activo: {query.isActive}")
    print(f"   ID: {query.id}")
    if query.isActive:
        print(f"   ✓ El streaming está CORRIENDO en background")
    else:
        print(f"   ✗ El streaming NO está activo (puede haber terminado o error)")
else:
    print("   ✗ No hay query definitida. Ejecuta la celda 3 primero.")

# 2. Verificar tabla Iceberg
print("\n📊 Contenido de la tabla Iceberg (lakehouse.bronze.payments_raw):")
try:
    total = spark.sql("SELECT COUNT(*) as total FROM lakehouse.bronze.payments_raw").collect()[0]["total"]
    print(f"   Total registros: {total}")
    
    if total > 0:
        print(f"\n   ✓ ¡ÉXITO! Se están ingiriendo eventos. Ultimos 3:")
        spark.sql("""
            SELECT event_time, payment_id, customer_id, status 
            FROM lakehouse.bronze.payments_raw 
            ORDER BY event_time DESC 
            LIMIT 3
        """).show(truncate=False)
    else:
        print("   ✗ La tabla está vacía. Revisa:")
        print("      1. ¿El generador está corriendo? (generar_datos.py)")
        print("      2. ¿Hay eventos en Kafka? (ejecuta la celda de DEBUG)")
        print("      3. ¿El parsing del JSON es correcto?")
except Exception as e:
    print(f"   ✗ Error al consultar tabla: {e}")


✅ VERIFICACIÓN FINAL DEL PROCESO

📡 Estado del Streaming Query:
   Activo: True
   ID: de754e50-a7d9-41f7-9093-742162aad247
   ✓ El streaming está CORRIENDO en background

📊 Contenido de la tabla Iceberg (lakehouse.bronze.payments_raw):
   Total registros: 42

   ✓ ¡ÉXITO! Se están ingiriendo eventos. Ultimos 3:
+---------------------------+------------------------------------+-----------+--------+
|event_time                 |payment_id                          |customer_id|status  |
+---------------------------+------------------------------------+-----------+--------+
|2026-04-12T23:37:05.915904Z|a2357424-ad10-4957-969e-da6e73c044f0|CUST_00050 |approved|
|2026-04-10T18:55:17.186501Z|d89113ea-b77a-429c-a4ed-54449b411efc|CUST_00048 |approved|
|2026-04-06T03:51:45.033146Z|19d63882-d941-45a8-8369-09978796446d|CUST_00032 |approved|
+---------------------------+------------------------------------+-----------+--------+

